In [16]:
# ============================================================
# Combined faceted simulation plot WITH stacked zooms
#
# Target size:
#   80 mm wide × 180 mm high
#
# Order:
#   skew, SNPs, escape, cells, depth
#
# Layout per simulation:
#   left  = main panel spanning two rows
#   right = accuracy zoom on top, gene zoom below
#
# Features:
#   - percentage version only
#   - accuracy and expressed chrX genes on twin y-axes
#   - secondary y-axis sits between main plot and zoom plots
#   - no secondary x-axis for skew
#   - secondary x-axis on main panels for SNPs / escape / cells / depth
#   - secondary x-axis on bottom zoom panels for SNPs / escape / cells / depth
#   - top zoom panels have no x-axis labels or secondary x-axis
#   - bottom zoom panels have primary x-axis labels
#   - zoom connector lines retained
#   - legend entries side-by-side
#   - both lines use circular points
#   - smaller spacer after skew because skew has no secondary x-axis
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle, ConnectionPatch

# ---------- paths ----------
gene_list_dir = Path(
    "/home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/processed_data/chrX_gene_read_cutoff_lists"
)
ge10_file = gene_list_dir / "chrX_gene_names_total_reads_ge10.txt"

plot_dir = Path(
    "/home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/simulation_combined/plots"
)
plot_dir.mkdir(parents=True, exist_ok=True)

formats = ["pdf", "svg", "png", "eps"]

# ---------- figure sizing ----------
MM_PER_INCH = 25.4

FIG_WIDTH_MM = 80
FIG_HEIGHT_MM = 180

FIGSIZE = (FIG_WIDTH_MM / MM_PER_INCH, FIG_HEIGHT_MM / MM_PER_INCH)

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 6.0,
    "axes.labelsize": 6.2,
    "axes.titlesize": 6.7,
    "xtick.labelsize": 5.4,
    "ytick.labelsize": 5.4,
    "legend.fontsize": 4.9,

    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.1,
    "ytick.major.size": 2.1,
    "lines.linewidth": 1.0,
    "lines.markersize": 2.6,
})

# ---------- colours ----------
acc_col = "#E76F51"
genes_col = "#2A9D8F"


def save_all(fig, name):
    for fmt in formats:
        fig.savefig(
            plot_dir / f"{name}.{fmt}",
            dpi=300,
            bbox_inches="tight",
            transparent=True,
        )
    plt.close(fig)


def format_reads(v):
    v = float(v)

    if v >= 1_000_000:
        return f"{v / 1_000_000:.1f}M"

    if v >= 1_000:
        return f"{v / 1_000:.0f}k"

    return f"{int(round(v))}"


# ---------- load denominator ----------
ge10_genes = pd.read_csv(ge10_file, header=None)[0].dropna().astype(str).unique()
n_chrX_ge10 = len(ge10_genes)

print(f"chrX genes with ≥10 reads: {n_chrX_ge10:,}")


# ---------- simulation definitions ----------
simulations = [
    {
        "key": "skew",
        "title": "Skew",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/skew"),
        "accuracy_file": "skew_accuracy_summary_by_run_stage.tsv",
        "x_col": "x1_fraction_setting",
        "x_label": "X1 fraction (%)",
        "x_transform": lambda s: s * 100,
        "extra_aggs": {},
        "x_ticks": [50, 75, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (50, 100),
    },
    {
        "key": "snps",
        "title": "SNP removal",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/snp_removal2"),
        "accuracy_file": "snp_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "snp_removal_pct_setting",
        "x_label": "SNPs removed (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_n_snps": ("n_snps", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
    {
        "key": "escape",
        "title": "Escape",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/escape"),
        "accuracy_file": "escape_accuracy_summary_by_run_stage.tsv",
        "x_col": "escape_pct_setting",
        "x_label": "Escape genes (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_n_escape_genes": ("n_escape_genes", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
    {
        "key": "cells",
        "title": "Cell removal",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/cell_removal"),
        "accuracy_file": "cell_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "cell_removal_pct_setting",
        "x_label": "Cells removed (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_n_cells_after_cell_removal": ("n_cells_after_cell_removal", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
    {
        "key": "depth",
        "title": "Depth",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/read_removal"),
        "accuracy_file": "read_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "read_removal_pct_setting",
        "x_label": "Reads removed (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_allele_reads_before": ("allele_reads_before_read_removal", "mean"),
            "mean_allele_reads_after": ("allele_reads_after_read_removal", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
]


def load_simulation_summary(config):
    accuracy_file = config["base"] / config["accuracy_file"]
    accuracy = pd.read_csv(accuracy_file, sep="\t")

    agg_dict = {
        "mean_phased_genes": ("n_genes", "mean"),
        "sd_phased_genes": ("n_genes", "std"),
        "mean_accuracy": ("accuracy", "mean"),
        "sd_accuracy": ("accuracy", "std"),
    }
    agg_dict.update(config["extra_aggs"])

    df = (
        accuracy[accuracy["Stage"].astype(str).eq("all")]
        .groupby(config["x_col"], as_index=False)
        .agg(**agg_dict)
        .sort_values(config["x_col"])
    )

    df["x_plot"] = config["x_transform"](df[config["x_col"]])

    df["mean_phased_gene_pct_chrX_ge10"] = (
        df["mean_phased_genes"] / n_chrX_ge10 * 100
    )
    df["sd_phased_gene_pct_chrX_ge10"] = (
        df["sd_phased_genes"] / n_chrX_ge10 * 100
    )

    df = df.fillna({
        "sd_accuracy": 0,
        "sd_phased_genes": 0,
        "sd_phased_gene_pct_chrX_ge10": 0,
    })

    df["acc_y"] = df["mean_accuracy"] * 100
    df["acc_sd"] = df["sd_accuracy"] * 100

    df["gene_y"] = df["mean_phased_gene_pct_chrX_ge10"]
    df["gene_sd"] = df["sd_phased_gene_pct_chrX_ge10"]

    return df


def get_secondary_labels(tick_df, config):
    key = config["key"]

    if key == "snps":
        labels = [f"{int(round(v)):,}" for v in tick_df["mean_n_snps"]]
        label = "Mean SNPs retained"

    elif key == "escape":
        labels = [f"{int(round(v)):,}" for v in tick_df["mean_n_escape_genes"]]
        label = "Mean escape genes"

    elif key == "cells":
        labels = [
            f"{int(round(v)):,}"
            for v in tick_df["mean_n_cells_after_cell_removal"]
        ]
        label = "Mean cells retained"

    elif key == "depth":
        labels = [format_reads(v) for v in tick_df["mean_allele_reads_after"]]
        label = "Allele reads retained"

    else:
        labels = [str(v) for v in tick_df["x_plot"]]
        label = ""

    return labels, label


def add_secondary_axis_main(ax, df, config):
    if config["key"] == "skew":
        return None

    secax = ax.secondary_xaxis("bottom")

    ticks = config["x_ticks"]
    tick_df = (
        df.iloc[[np.argmin(np.abs(df["x_plot"] - t)) for t in ticks]]
        .drop_duplicates("x_plot")
        .sort_values("x_plot")
    )

    secax.set_xticks(tick_df["x_plot"])

    labels, label = get_secondary_labels(tick_df, config)

    secax.set_xticklabels(labels)
    secax.set_xlabel(label, fontsize=5.1, labelpad=1.8)
    secax.spines["bottom"].set_position(("outward", 20))
    secax.tick_params(
        axis="x",
        labelsize=4.6,
        pad=1.1,
        length=1.6,
        width=0.45,
    )

    return secax


def add_secondary_axis_zoom(ax, zoom_df, config):
    if config["key"] == "skew":
        return None

    secax = ax.secondary_xaxis("bottom")

    ticks = config["zoom_ticks"]
    tick_df = (
        zoom_df.iloc[[np.argmin(np.abs(zoom_df["x_plot"] - t)) for t in ticks]]
        .drop_duplicates("x_plot")
        .sort_values("x_plot")
    )

    secax.set_xticks(tick_df["x_plot"])

    labels, label = get_secondary_labels(tick_df, config)

    secax.set_xticklabels(labels)
    secax.set_xlabel(label, fontsize=4.6, labelpad=1.3)
    secax.spines["bottom"].set_position(("outward", 13))
    secax.tick_params(
        axis="x",
        labelsize=4.2,
        pad=0.9,
        length=1.4,
        width=0.4,
    )

    return secax


def nice_zoom_limits(y, y_sd, lower_bound=0, upper_bound=100, step=5):
    ymin = max(lower_bound, np.floor((y - y_sd).min() / step) * step)
    ymax = min(upper_bound, np.ceil((y + y_sd).max() / step) * step)

    if ymin == ymax:
        ymin = max(lower_bound, ymin - step / 2)
        ymax = min(upper_bound, ymax + step / 2)

    return ymin, ymax


def plot_faceted_with_stacked_zooms():
    loaded = [(config, load_simulation_summary(config)) for config in simulations]

    fig = plt.figure(figsize=FIGSIZE)

    # ------------------------------------------------------------
    # Each simulation uses:
    #   row r0     = accuracy zoom
    #   row r0 + 1 = gene zoom
    #   row r0 + 2 = spacer before next simulation
    #
    # Main panel spans r0 and r0 + 1.
    #
    # The skew spacer is smaller because skew has no secondary x-axis.
    # ------------------------------------------------------------
    height_ratios = []

    for i in range(len(simulations)):
        height_ratios.extend([1.0, 1.0])

        if i < len(simulations) - 1:
            if simulations[i]["key"] == "skew":
                height_ratios.append(0.48)   # smaller gap after skew, no secondary x-axis
            else:
                height_ratios.append(0.72)   # slightly more space between other plots

    gs = GridSpec(
        nrows=len(height_ratios),
        ncols=2,
        height_ratios=height_ratios,
        width_ratios=[2.65, 1.0],
        hspace=0.40,
        wspace=0.74,
    )

    legend_handles = None
    legend_labels = None

    for i, (config, df) in enumerate(loaded):
        r0 = i * 3
        r1 = r0 + 1

        ax_main = fig.add_subplot(gs[r0:r1 + 1, 0])
        ax_main_gene = ax_main.twinx()

        ax_zoom_acc = fig.add_subplot(gs[r0, 1])
        ax_zoom_gene = fig.add_subplot(gs[r1, 1])

        x = df["x_plot"]

        acc = df["acc_y"]
        acc_sd = df["acc_sd"]

        gene = df["gene_y"]
        gene_sd = df["gene_sd"]

        # ---------- main panel ----------
        ax_main.plot(
            x,
            acc,
            marker="o",
            color=acc_col,
            label="Accuracy (%)",
        )
        ax_main.fill_between(
            x,
            acc - acc_sd,
            acc + acc_sd,
            color=acc_col,
            alpha=0.18,
            linewidth=0,
        )

        ax_main_gene.plot(
            x,
            gene,
            marker="o",
            linestyle="--",
            color=genes_col,
            label="Expressed chrX genes phased (%)",
        )
        ax_main_gene.fill_between(
            x,
            gene - gene_sd,
            gene + gene_sd,
            color=genes_col,
            alpha=0.14,
            linewidth=0,
        )

        ax_main.set_xlim(*config["xlim"])
        ax_main.set_ylim(0, 100)
        ax_main_gene.set_ylim(0, 100)

        ax_main.set_xticks(config["x_ticks"])
        ax_main.set_xlabel(config["x_label"], labelpad=2.0, fontsize=5.7)

        ax_main.set_title(
            config["title"],
            loc="left",
            fontweight="bold",
            pad=1.6,
            fontsize=6.4,
        )

        add_secondary_axis_main(ax_main, df, config)

        # ---------- zoom data ----------
        zoom_df = df[df["x_plot"].between(99, 100)].copy()

        if zoom_df.empty:
            ax_zoom_acc.set_visible(False)
            ax_zoom_gene.set_visible(False)

        else:
            xz = zoom_df["x_plot"]

            acc_z = zoom_df["acc_y"]
            acc_z_sd = zoom_df["acc_sd"]

            gene_z = zoom_df["gene_y"]
            gene_z_sd = zoom_df["gene_sd"]

            acc_ymin, acc_ymax = nice_zoom_limits(
                acc_z,
                acc_z_sd,
                lower_bound=0,
                upper_bound=100,
                step=5,
            )

            gene_ymin, gene_ymax = nice_zoom_limits(
                gene_z,
                gene_z_sd,
                lower_bound=0,
                upper_bound=100,
                step=5,
            )

            # ---------- highlight zoom regions on main ----------
            ax_main.add_patch(
                Rectangle(
                    (99, acc_ymin),
                    1,
                    acc_ymax - acc_ymin,
                    facecolor=acc_col,
                    edgecolor=acc_col,
                    alpha=0.10,
                    linewidth=0.6,
                    zorder=0.5,
                )
            )

            ax_main_gene.add_patch(
                Rectangle(
                    (99, gene_ymin),
                    1,
                    gene_ymax - gene_ymin,
                    facecolor=genes_col,
                    edgecolor=genes_col,
                    alpha=0.10,
                    linewidth=0.6,
                    transform=ax_main_gene.transData,
                    zorder=0.5,
                )
            )

            # ---------- accuracy zoom ----------
            ax_zoom_acc.plot(
                xz,
                acc_z,
                marker="o",
                color=acc_col,
            )
            ax_zoom_acc.fill_between(
                xz,
                acc_z - acc_z_sd,
                acc_z + acc_z_sd,
                color=acc_col,
                alpha=0.18,
                linewidth=0,
            )

            ax_zoom_acc.set_xlim(98.95, 100.02)
            ax_zoom_acc.set_ylim(acc_ymin, acc_ymax)
            ax_zoom_acc.set_xticks(config["zoom_ticks"])

            # Top zoom: no x tick labels, no primary x label, no secondary x-axis
            ax_zoom_acc.set_xlabel("")
            ax_zoom_acc.tick_params(
                axis="x",
                labelbottom=False,
                bottom=True,
                length=1.8,
                width=0.5,
            )

            ax_zoom_acc.set_ylabel(
                "Accuracy (%)",
                color=acc_col,
                fontsize=5.4,
                labelpad=1.0,
            )
            ax_zoom_acc.tick_params(
                axis="y",
                labelcolor=acc_col,
                labelsize=4.8,
                pad=1.0,
                width=0.5,
                length=1.8,
            )

            # ---------- gene zoom ----------
            ax_zoom_gene.plot(
                xz,
                gene_z,
                marker="o",
                linestyle="--",
                color=genes_col,
            )
            ax_zoom_gene.fill_between(
                xz,
                gene_z - gene_z_sd,
                gene_z + gene_z_sd,
                color=genes_col,
                alpha=0.14,
                linewidth=0,
            )

            ax_zoom_gene.set_xlim(98.95, 100.02)
            ax_zoom_gene.set_ylim(gene_ymin, gene_ymax)
            ax_zoom_gene.set_xticks(config["zoom_ticks"])

            ax_zoom_gene.set_xlabel(
                config["x_label"],
                fontsize=5.1,
                labelpad=1.2,
            )
            ax_zoom_gene.tick_params(
                axis="x",
                labelsize=4.6,
                pad=0.9,
                width=0.5,
                length=1.8,
            )

            ax_zoom_gene.set_ylabel(
                "Genes (%)",
                color=genes_col,
                fontsize=5.4,
                labelpad=1.0,
            )
            ax_zoom_gene.tick_params(
                axis="y",
                labelcolor=genes_col,
                labelsize=4.8,
                pad=1.0,
                width=0.5,
                length=1.8,
            )

            add_secondary_axis_zoom(ax_zoom_gene, zoom_df, config)

            # ---------- connector lines ----------
            for y_main, y_zoom in [
                (acc_ymin, ax_zoom_acc.get_ylim()[0]),
                (acc_ymax, ax_zoom_acc.get_ylim()[1]),
            ]:
                con = ConnectionPatch(
                    xyA=(100, y_main),
                    coordsA=ax_main.transData,
                    xyB=(98.95, y_zoom),
                    coordsB=ax_zoom_acc.transData,
                    color="0.35",
                    alpha=0.30,
                    linewidth=0.5,
                    zorder=0,
                    clip_on=False,
                )
                fig.add_artist(con)

            for y_main, y_zoom in [
                (gene_ymin, ax_zoom_gene.get_ylim()[0]),
                (gene_ymax, ax_zoom_gene.get_ylim()[1]),
            ]:
                con = ConnectionPatch(
                    xyA=(100, y_main),
                    coordsA=ax_main_gene.transData,
                    xyB=(98.95, y_zoom),
                    coordsB=ax_zoom_gene.transData,
                    color="0.35",
                    alpha=0.30,
                    linewidth=0.5,
                    zorder=0,
                    clip_on=False,
                )
                fig.add_artist(con)

        # ---------- styling ----------
        for ax in [ax_main, ax_zoom_acc, ax_zoom_gene]:
            ax.spines["top"].set_visible(False)
            ax.grid(axis="y", alpha=0.25, linewidth=0.5)
            ax.set_axisbelow(True)

        ax_main_gene.spines["top"].set_visible(False)

        ax_main.tick_params(
            axis="y",
            labelcolor=acc_col,
            labelsize=5.4,
            pad=1.2,
            width=0.55,
            length=2.0,
        )
        ax_main_gene.tick_params(
            axis="y",
            labelcolor=genes_col,
            labelsize=5.4,
            pad=1.2,
            width=0.55,
            length=2.0,
        )
        ax_main.tick_params(
            axis="x",
            labelsize=5.4,
            pad=1.2,
            width=0.55,
            length=2.0,
        )

        if i == 2:
            ax_main.set_ylabel(
                "Accuracy (%)",
                color=acc_col,
                fontsize=6.4,
                fontweight="bold",
                labelpad=5.5,
            )
            ax_main_gene.set_ylabel(
                "Expressed chrX genes phased (%)",
                color=genes_col,
                fontsize=6.0,
                fontweight="bold",
                rotation=270,
                labelpad=4.0,
            )
        else:
            ax_main.set_ylabel("")
            ax_main_gene.set_ylabel("")

        if legend_handles is None:
            h1, l1 = ax_main.get_legend_handles_labels()
            h2, l2 = ax_main_gene.get_legend_handles_labels()
            legend_handles = h1 + h2
            legend_labels = l1 + l2

    # ---------- legend ----------
    fig.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.994),
        ncol=2,
        frameon=False,
        handlelength=1.2,
        columnspacing=0.9,
        labelspacing=0.25,
        fontsize=4.9,
    )

    fig.subplots_adjust(
        left=0.11,
        right=0.97,
        top=0.955,
        bottom=0.04,
    )

    save_all(
        fig,
        "combined_simulations_faceted_stacked_zooms_80x180mm_percentage_sd",
    )


# ---------- make figure ----------
plot_faceted_with_stacked_zooms()

print(f"Wrote combined stacked-zoom plot to: {plot_dir}")

chrX genes with ≥10 reads: 489


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote combined stacked-zoom plot to: /home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/simulation_combined/plots


In [6]:
# ============================================================
# Combined faceted simulation plots WITH stacked zooms
#
# Horizontal version split into:
#   1) skew, SNPs, escape
#   2) cells, depth
#
# Layout per simulation/facet:
#   left  = main panel spanning two rows
#   right = accuracy zoom on top, gene zoom below
#
# Features:
#   - percentage version only
#   - accuracy and expressed chrX genes on twin y-axes
#   - no secondary x-axis for skew
#   - secondary x-axis on main panels for SNPs / escape / cells / depth
#   - secondary x-axis on bottom zoom panels for SNPs / escape / cells / depth
#   - top zoom panels have no x-axis labels or secondary x-axis
#   - bottom zoom panels have primary x-axis labels
#   - zoom connector lines retained
#   - legend entries side-by-side
#   - both lines use circular points
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle, ConnectionPatch

# ---------- paths ----------
gene_list_dir = Path(
    "/home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/processed_data/chrX_gene_read_cutoff_lists"
)
ge10_file = gene_list_dir / "chrX_gene_names_total_reads_ge10.txt"

plot_dir = Path(
    "/home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/simulation_combined/plots"
)
plot_dir.mkdir(parents=True, exist_ok=True)

formats = ["pdf", "svg", "png", "eps"]


# ---------- figure sizing ----------
# ---------- figure sizing ----------
MM_PER_INCH = 25.4

FACET_WIDTH_MM = 75
FIG_HEIGHT_MM = 60

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 6.0,
    "axes.labelsize": 6.2,
    "axes.titlesize": 6.7,
    "xtick.labelsize": 5.4,
    "ytick.labelsize": 5.4,
    "legend.fontsize": 4.9,

    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.1,
    "ytick.major.size": 2.1,
    "lines.linewidth": 1.0,
    "lines.markersize": 2.6,
})

# ---------- colours ----------
acc_col = "#E76F51"
genes_col = "#2A9D8F"


def save_all(fig, name):
    for fmt in formats:
        fig.savefig(
            plot_dir / f"{name}.{fmt}",
            dpi=300,
            bbox_inches="tight",
            transparent=True,
        )
    plt.close(fig)


def format_reads(v):
    v = float(v)

    if v >= 1_000_000:
        return f"{v / 1_000_000:.1f}M"

    if v >= 1_000:
        return f"{v / 1_000:.0f}k"

    return f"{int(round(v))}"


# ---------- load denominator ----------
ge10_genes = pd.read_csv(ge10_file, header=None)[0].dropna().astype(str).unique()
n_chrX_ge10 = len(ge10_genes)

print(f"chrX genes with ≥10 reads: {n_chrX_ge10:,}")


# ---------- simulation definitions ----------
simulations = [
    {
        "key": "skew",
        "title": "Skew",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/skew"),
        "accuracy_file": "skew_accuracy_summary_by_run_stage.tsv",
        "x_col": "x1_fraction_setting",
        "x_label": "X1 fraction (%)",
        "x_transform": lambda s: s * 100,
        "extra_aggs": {},
        "x_ticks": [50, 75, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (50, 100),
    },
    {
        "key": "snps",
        "title": "SNP removal",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/snp_removal2"),
        "accuracy_file": "snp_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "snp_removal_pct_setting",
        "x_label": "SNPs removed (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_n_snps": ("n_snps", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
    {
        "key": "escape",
        "title": "Escape",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/escape"),
        "accuracy_file": "escape_accuracy_summary_by_run_stage.tsv",
        "x_col": "escape_pct_setting",
        "x_label": "Escape genes (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_n_escape_genes": ("n_escape_genes", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
    {
        "key": "cells",
        "title": "Cell removal",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/cell_removal"),
        "accuracy_file": "cell_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "cell_removal_pct_setting",
        "x_label": "Cells removed (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_n_cells_after_cell_removal": ("n_cells_after_cell_removal", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
    {
        "key": "depth",
        "title": "Depth",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/read_removal"),
        "accuracy_file": "read_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "read_removal_pct_setting",
        "x_label": "Reads removed (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_allele_reads_before": ("allele_reads_before_read_removal", "mean"),
            "mean_allele_reads_after": ("allele_reads_after_read_removal", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
]


def load_simulation_summary(config):
    accuracy_file = config["base"] / config["accuracy_file"]
    accuracy = pd.read_csv(accuracy_file, sep="\t")

    agg_dict = {
        "mean_phased_genes": ("n_genes", "mean"),
        "sd_phased_genes": ("n_genes", "std"),
        "mean_accuracy": ("accuracy", "mean"),
        "sd_accuracy": ("accuracy", "std"),
    }
    agg_dict.update(config["extra_aggs"])

    df = (
        accuracy[accuracy["Stage"].astype(str).eq("all")]
        .groupby(config["x_col"], as_index=False)
        .agg(**agg_dict)
        .sort_values(config["x_col"])
    )

    df["x_plot"] = config["x_transform"](df[config["x_col"]])

    df["mean_phased_gene_pct_chrX_ge10"] = (
        df["mean_phased_genes"] / n_chrX_ge10 * 100
    )
    df["sd_phased_gene_pct_chrX_ge10"] = (
        df["sd_phased_genes"] / n_chrX_ge10 * 100
    )

    df = df.fillna({
        "sd_accuracy": 0,
        "sd_phased_genes": 0,
        "sd_phased_gene_pct_chrX_ge10": 0,
    })

    df["acc_y"] = df["mean_accuracy"] * 100
    df["acc_sd"] = df["sd_accuracy"] * 100

    df["gene_y"] = df["mean_phased_gene_pct_chrX_ge10"]
    df["gene_sd"] = df["sd_phased_gene_pct_chrX_ge10"]

    return df


def get_secondary_labels(tick_df, config):
    key = config["key"]

    if key == "snps":
        labels = [f"{int(round(v)):,}" for v in tick_df["mean_n_snps"]]
        label = "Mean SNPs retained"

    elif key == "escape":
        labels = [f"{int(round(v)):,}" for v in tick_df["mean_n_escape_genes"]]
        label = "Mean escape genes"

    elif key == "cells":
        labels = [
            f"{int(round(v)):,}"
            for v in tick_df["mean_n_cells_after_cell_removal"]
        ]
        label = "Mean cells retained"

    elif key == "depth":
        labels = [format_reads(v) for v in tick_df["mean_allele_reads_after"]]
        label = "Allele reads retained"

    else:
        labels = [str(v) for v in tick_df["x_plot"]]
        label = ""

    return labels, label


def add_secondary_axis_main(ax, df, config):
    if config["key"] == "skew":
        return None

    secax = ax.secondary_xaxis("bottom")

    ticks = config["x_ticks"]
    tick_df = (
        df.iloc[[np.argmin(np.abs(df["x_plot"] - t)) for t in ticks]]
        .drop_duplicates("x_plot")
        .sort_values("x_plot")
    )

    secax.set_xticks(tick_df["x_plot"])

    labels, label = get_secondary_labels(tick_df, config)

    secax.set_xticklabels(labels)
    secax.set_xlabel(label, fontsize=4.8, labelpad=1.6)
    secax.spines["bottom"].set_position(("outward", 18))
    secax.tick_params(
        axis="x",
        labelsize=4.3,
        pad=1.0,
        length=1.5,
        width=0.45,
    )

    return secax


def add_secondary_axis_zoom(ax, zoom_df, config):
    if config["key"] == "skew":
        return None

    secax = ax.secondary_xaxis("bottom")

    ticks = config["zoom_ticks"]
    tick_df = (
        zoom_df.iloc[[np.argmin(np.abs(zoom_df["x_plot"] - t)) for t in ticks]]
        .drop_duplicates("x_plot")
        .sort_values("x_plot")
    )

    secax.set_xticks(tick_df["x_plot"])

    labels, label = get_secondary_labels(tick_df, config)

    secax.set_xticklabels(labels)
    secax.set_xlabel(label, fontsize=4.3, labelpad=1.1)
    secax.spines["bottom"].set_position(("outward", 12))
    secax.tick_params(
        axis="x",
        labelsize=3.9,
        pad=0.8,
        length=1.3,
        width=0.4,
    )

    return secax


def nice_zoom_limits(y, y_sd, lower_bound=0, upper_bound=100, step=5):
    ymin = max(lower_bound, np.floor((y - y_sd).min() / step) * step)
    ymax = min(upper_bound, np.ceil((y + y_sd).max() / step) * step)

    if ymin == ymax:
        ymin = max(lower_bound, ymin - step / 2)
        ymax = min(upper_bound, ymax + step / 2)

    return ymin, ymax


def plot_one_facet(
    fig,
    subgs,
    config,
    df,
    facet_index,
    n_facets,
):
    ax_main = fig.add_subplot(subgs[:, 0])
    ax_main_gene = ax_main.twinx()

    ax_zoom_acc = fig.add_subplot(subgs[0, 1])
    ax_zoom_gene = fig.add_subplot(subgs[1, 1])

    x = df["x_plot"]

    acc = df["acc_y"]
    acc_sd = df["acc_sd"]

    gene = df["gene_y"]
    gene_sd = df["gene_sd"]

    # ---------- main panel ----------
    ax_main.plot(
        x,
        acc,
        marker="o",
        color=acc_col,
        label="Accuracy (%)",
    )
    ax_main.fill_between(
        x,
        acc - acc_sd,
        acc + acc_sd,
        color=acc_col,
        alpha=0.18,
        linewidth=0,
    )

    ax_main_gene.plot(
        x,
        gene,
        marker="o",
        linestyle="--",
        color=genes_col,
        label="Expressed chrX genes phased (%)",
    )
    ax_main_gene.fill_between(
        x,
        gene - gene_sd,
        gene + gene_sd,
        color=genes_col,
        alpha=0.14,
        linewidth=0,
    )

    ax_main.set_xlim(*config["xlim"])
    ax_main.set_ylim(0, 100)
    ax_main_gene.set_ylim(0, 100)

    ax_main.set_xticks(config["x_ticks"])
    ax_main.set_xlabel(config["x_label"], labelpad=1.8, fontsize=5.4)

    ax_main.set_title(
        config["title"],
        loc="left",
        fontweight="bold",
        pad=1.6,
        fontsize=6.4,
    )

    add_secondary_axis_main(ax_main, df, config)

    # ---------- zoom data ----------
    zoom_df = df[df["x_plot"].between(99, 100)].copy()

    if zoom_df.empty:
        ax_zoom_acc.set_visible(False)
        ax_zoom_gene.set_visible(False)

    else:
        xz = zoom_df["x_plot"]

        acc_z = zoom_df["acc_y"]
        acc_z_sd = zoom_df["acc_sd"]

        gene_z = zoom_df["gene_y"]
        gene_z_sd = zoom_df["gene_sd"]

        acc_ymin, acc_ymax = nice_zoom_limits(
            acc_z,
            acc_z_sd,
            lower_bound=0,
            upper_bound=100,
            step=5,
        )

        gene_ymin, gene_ymax = nice_zoom_limits(
            gene_z,
            gene_z_sd,
            lower_bound=0,
            upper_bound=100,
            step=5,
        )

        # ---------- highlight zoom regions on main ----------
        ax_main.add_patch(
            Rectangle(
                (99, acc_ymin),
                1,
                acc_ymax - acc_ymin,
                facecolor=acc_col,
                edgecolor=acc_col,
                alpha=0.10,
                linewidth=0.6,
                zorder=0.5,
            )
        )

        ax_main_gene.add_patch(
            Rectangle(
                (99, gene_ymin),
                1,
                gene_ymax - gene_ymin,
                facecolor=genes_col,
                edgecolor=genes_col,
                alpha=0.10,
                linewidth=0.6,
                transform=ax_main_gene.transData,
                zorder=0.5,
            )
        )

        # ---------- accuracy zoom ----------
        ax_zoom_acc.plot(
            xz,
            acc_z,
            marker="o",
            color=acc_col,
        )
        ax_zoom_acc.fill_between(
            xz,
            acc_z - acc_z_sd,
            acc_z + acc_z_sd,
            color=acc_col,
            alpha=0.18,
            linewidth=0,
        )

        ax_zoom_acc.set_xlim(98.95, 100.02)
        ax_zoom_acc.set_ylim(acc_ymin, acc_ymax)
        ax_zoom_acc.set_xticks(config["zoom_ticks"])

        # Top zoom: no x tick labels, no primary x label, no secondary x-axis
        ax_zoom_acc.set_xlabel("")
        ax_zoom_acc.tick_params(
            axis="x",
            labelbottom=False,
            bottom=True,
            length=1.7,
            width=0.5,
        )

        ax_zoom_acc.set_ylabel(
            "Accuracy (%)",
            color=acc_col,
            fontsize=5.0,
            labelpad=0.8,
        )
        ax_zoom_acc.tick_params(
            axis="y",
            labelcolor=acc_col,
            labelsize=4.5,
            pad=0.8,
            width=0.5,
            length=1.7,
        )

        # ---------- gene zoom ----------
        ax_zoom_gene.plot(
            xz,
            gene_z,
            marker="o",
            linestyle="--",
            color=genes_col,
        )
        ax_zoom_gene.fill_between(
            xz,
            gene_z - gene_z_sd,
            gene_z + gene_z_sd,
            color=genes_col,
            alpha=0.14,
            linewidth=0,
        )

        ax_zoom_gene.set_xlim(98.95, 100.02)
        ax_zoom_gene.set_ylim(gene_ymin, gene_ymax)
        ax_zoom_gene.set_xticks(config["zoom_ticks"])

        ax_zoom_gene.set_xlabel(
            config["x_label"],
            fontsize=4.8,
            labelpad=1.0,
        )
        ax_zoom_gene.tick_params(
            axis="x",
            labelsize=4.2,
            pad=0.8,
            width=0.5,
            length=1.7,
        )

        ax_zoom_gene.set_ylabel(
            "Genes (%)",
            color=genes_col,
            fontsize=5.0,
            labelpad=0.8,
        )
        ax_zoom_gene.tick_params(
            axis="y",
            labelcolor=genes_col,
            labelsize=4.5,
            pad=0.8,
            width=0.5,
            length=1.7,
        )

        add_secondary_axis_zoom(ax_zoom_gene, zoom_df, config)

        # ---------- connector lines ----------
        for y_main, y_zoom in [
            (acc_ymin, ax_zoom_acc.get_ylim()[0]),
            (acc_ymax, ax_zoom_acc.get_ylim()[1]),
        ]:
            con = ConnectionPatch(
                xyA=(100, y_main),
                coordsA=ax_main.transData,
                xyB=(98.95, y_zoom),
                coordsB=ax_zoom_acc.transData,
                color="0.35",
                alpha=0.30,
                linewidth=0.5,
                zorder=0,
                clip_on=False,
            )
            fig.add_artist(con)

        for y_main, y_zoom in [
            (gene_ymin, ax_zoom_gene.get_ylim()[0]),
            (gene_ymax, ax_zoom_gene.get_ylim()[1]),
        ]:
            con = ConnectionPatch(
                xyA=(100, y_main),
                coordsA=ax_main_gene.transData,
                xyB=(98.95, y_zoom),
                coordsB=ax_zoom_gene.transData,
                color="0.35",
                alpha=0.30,
                linewidth=0.5,
                zorder=0,
                clip_on=False,
            )
            fig.add_artist(con)

    # ---------- styling ----------
    for ax in [ax_main, ax_zoom_acc, ax_zoom_gene]:
        ax.spines["top"].set_visible(False)
        ax.grid(axis="y", alpha=0.25, linewidth=0.5)
        ax.set_axisbelow(True)

    ax_main_gene.spines["top"].set_visible(False)

    ax_main.tick_params(
        axis="y",
        labelcolor=acc_col,
        labelsize=5.0,
        pad=1.0,
        width=0.55,
        length=2.0,
    )
    ax_main_gene.tick_params(
        axis="y",
        labelcolor=genes_col,
        labelsize=5.0,
        pad=1.0,
        width=0.55,
        length=2.0,
    )
    ax_main.tick_params(
        axis="x",
        labelsize=5.0,
        pad=1.0,
        width=0.55,
        length=2.0,
    )

    # Only put the big main y-axis labels at the outside of each horizontal figure
    if facet_index == 0:
        ax_main.set_ylabel(
            "Accuracy (%)",
            color=acc_col,
            fontsize=6.0,
            fontweight="bold",
            labelpad=4.0,
        )
    else:
        ax_main.set_ylabel("")

    if facet_index == n_facets - 1:
        ax_main_gene.set_ylabel(
            "Expressed chrX genes phased (%)",
            color=genes_col,
            fontsize=5.7,
            fontweight="bold",
            rotation=270,
            labelpad=2.0,
        )
    else:
        ax_main_gene.set_ylabel("")

    return ax_main, ax_main_gene


def plot_horizontal_group(configs, output_name):
    loaded = [(config, load_simulation_summary(config)) for config in configs]

    n_facets = len(loaded)

    facet_width_mm = 75
    fig_width_mm = facet_width_mm * n_facets
    fig_height_mm = FIG_HEIGHT_MM

    fig = plt.figure(
        figsize=(fig_width_mm / MM_PER_INCH, fig_height_mm / MM_PER_INCH)
    )

    outer_gs = GridSpec(
        nrows=1,
        ncols=n_facets,
        figure=fig,
        wspace=0.08,
    )

    legend_handles = None
    legend_labels = None

    for i, (config, df) in enumerate(loaded):
        # Give the last facet a bit more room between main plot and zooms
        facet_wspace = 0.56 if i == n_facets - 1 else 0.42

        subgs = outer_gs[0, i].subgridspec(
            nrows=2,
            ncols=2,
            height_ratios=[1.0, 1.0],
            width_ratios=[2.35, 1.35],
            hspace=0.24,
            wspace=facet_wspace,
        )

        ax_main, ax_main_gene = plot_one_facet(
            fig=fig,
            subgs=subgs,
            config=config,
            df=df,
            facet_index=i,
            n_facets=n_facets,
        )

        if legend_handles is None:
            h1, l1 = ax_main.get_legend_handles_labels()
            h2, l2 = ax_main_gene.get_legend_handles_labels()
            legend_handles = h1 + h2
            legend_labels = l1 + l2

    fig.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.995),
        ncol=2,
        frameon=False,
        handlelength=1.2,
        columnspacing=0.9,
        labelspacing=0.25,
        fontsize=4.9,
    )

    left = 0.045 if n_facets >= 3 else 0.060

    fig.subplots_adjust(
        left=left,
        right=0.985,
        top=0.830,
        bottom=0.260,
    )

    save_all(fig, output_name)

    print(
        f"Wrote {n_facets}-facet horizontal plot "
        f"({fig_width_mm} × {fig_height_mm} mm) to: {plot_dir / output_name}"
    )


# ---------- make figures ----------
plot_horizontal_group(
    simulations[:3],
    "combined_simulations_horizontal_skew_snps_escape_180x60mm_percentage_sd",
)

plot_horizontal_group(
    simulations[3:],
    "combined_simulations_horizontal_cells_depth_120x60mm_percentage_sd",
)

print(f"Done. Wrote horizontal split plots to: {plot_dir}")

chrX genes with ≥10 reads: 489


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote 3-facet horizontal plot (225 × 60 mm) to: /home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/simulation_combined/plots/combined_simulations_horizontal_skew_snps_escape_180x60mm_percentage_sd


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote 2-facet horizontal plot (150 × 60 mm) to: /home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/simulation_combined/plots/combined_simulations_horizontal_cells_depth_120x60mm_percentage_sd
Done. Wrote horizontal split plots to: /home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/simulation_combined/plots


In [1]:
# ============================================================
# Combined faceted simulation plot:
# accuracy per stage + overall across sweeps
#
# Panels:
#   skew, SNPs, escape, cells, depth
#
# Each simulation:
#   left  = full sweep accuracy
#   right = zoom of 99–100% sweep region
#
# Lines:
#   Stage 1
#   Stage 2
#   Overall
#
# Accuracy is plotted as mean ± SD across runs.
# ============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle, ConnectionPatch
from matplotlib.lines import Line2D


# ============================================================
# Paths
# ============================================================

plot_dir = Path(
    "/home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/simulation_combined/plots"
)
plot_dir.mkdir(parents=True, exist_ok=True)

formats = ["pdf", "svg", "png", "eps"]


# ============================================================
# Figure sizing
# ============================================================

MM_PER_INCH = 25.4

FIG_WIDTH_MM = 80
FIG_HEIGHT_MM = 180

FIGSIZE = (FIG_WIDTH_MM / MM_PER_INCH, FIG_HEIGHT_MM / MM_PER_INCH)


# ============================================================
# Matplotlib style
# ============================================================

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 6.0,
    "axes.labelsize": 6.2,
    "axes.titlesize": 6.7,
    "xtick.labelsize": 5.4,
    "ytick.labelsize": 5.4,
    "legend.fontsize": 5.0,

    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.1,
    "ytick.major.size": 2.1,
    "lines.linewidth": 1.0,
    "lines.markersize": 2.6,
})


# ============================================================
# Colours
# ============================================================

stage_order = ["Stage 1", "Stage 2", "Overall"]

stage_colours = {
    "Stage 1": "#4C78A8",
    "Stage 2": "#F58518",
    "Overall": "#E76F51",
}

stage_linestyles = {
    "Stage 1": "-",
    "Stage 2": "-",
    "Overall": "-",
}


# ============================================================
# Helpers
# ============================================================

def save_all(fig, name):
    for fmt in formats:
        fig.savefig(
            plot_dir / f"{name}.{fmt}",
            dpi=300,
            bbox_inches="tight",
            transparent=True,
        )
    plt.close(fig)


def format_reads(v):
    v = float(v)

    if v >= 1_000_000:
        return f"{v / 1_000_000:.1f}M"

    if v >= 1_000:
        return f"{v / 1_000:.0f}k"

    return f"{int(round(v))}"


def normalize_stage(stage):
    """
    Robustly convert stage labels to:
        Stage 1
        Stage 2
        Overall

    Handles examples like:
        1
        1.0
        Stage 1
        Stage 1.0
        stage_1
        all
        overall
    """

    s = str(stage).strip().lower()
    s = s.replace("_", " ").replace("-", " ")

    if s in {"all", "overall", "stage all"}:
        return "Overall"

    if s in {"1", "1.0"}:
        return "Stage 1"

    if s in {"2", "2.0"}:
        return "Stage 2"

    nums = re.findall(r"\d+(?:\.\d+)?", s)

    if "stage" in s and nums:
        stage_num = int(float(nums[0]))

        if stage_num == 1:
            return "Stage 1"

        if stage_num == 2:
            return "Stage 2"

    return str(stage)


def nice_zoom_limits(y, y_sd, lower_bound=0, upper_bound=100, step=1):
    """
    Make compact y-axis limits for the zoom panel.
    """

    y = pd.Series(y).astype(float)
    y_sd = pd.Series(y_sd).astype(float).fillna(0)

    ymin = max(lower_bound, np.floor((y - y_sd).min() / step) * step)
    ymax = min(upper_bound, np.ceil((y + y_sd).max() / step) * step)

    if not np.isfinite(ymin) or not np.isfinite(ymax):
        return lower_bound, upper_bound

    if ymin == ymax:
        ymin = max(lower_bound, ymin - step)
        ymax = min(upper_bound, ymax + step)

    if ymax - ymin < 2:
        mid = (ymax + ymin) / 2
        ymin = max(lower_bound, mid - 1)
        ymax = min(upper_bound, mid + 1)

    return ymin, ymax


# ============================================================
# Simulation definitions
# ============================================================

simulations = [
    {
        "key": "skew",
        "title": "Skew",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/skew"),
        "accuracy_file": "skew_accuracy_summary_by_run_stage.tsv",
        "x_col": "x1_fraction_setting",
        "x_label": "X1 fraction (%)",
        "x_transform": lambda s: s * 100,
        "extra_aggs": {},
        "x_ticks": [50, 75, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (50, 100),
    },
    {
        "key": "snps",
        "title": "SNP removal",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/snp_removal2"),
        "accuracy_file": "snp_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "snp_removal_pct_setting",
        "x_label": "SNPs removed (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_n_snps": ("n_snps", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
    {
        "key": "escape",
        "title": "Escape",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/escape"),
        "accuracy_file": "escape_accuracy_summary_by_run_stage.tsv",
        "x_col": "escape_pct_setting",
        "x_label": "Escape genes (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_n_escape_genes": ("n_escape_genes", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
    {
        "key": "cells",
        "title": "Cell removal",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/cell_removal"),
        "accuracy_file": "cell_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "cell_removal_pct_setting",
        "x_label": "Cells removed (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_n_cells_after_cell_removal": ("n_cells_after_cell_removal", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
    {
        "key": "depth",
        "title": "Depth",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/read_removal"),
        "accuracy_file": "read_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "read_removal_pct_setting",
        "x_label": "Reads removed (%)",
        "x_transform": lambda s: s,
        "extra_aggs": {
            "mean_allele_reads_before": ("allele_reads_before_read_removal", "mean"),
            "mean_allele_reads_after": ("allele_reads_after_read_removal", "mean"),
        },
        "x_ticks": [0, 50, 100],
        "zoom_ticks": [99.0, 99.5, 99.9],
        "xlim": (0, 100),
    },
]


# ============================================================
# Loading and aggregation
# ============================================================

def load_stage_accuracy_summary(config):
    accuracy_file = config["base"] / config["accuracy_file"]

    print(f"Reading: {accuracy_file}")

    accuracy = pd.read_csv(accuracy_file, sep="\t")

    required_cols = {config["x_col"], "Stage", "accuracy"}
    missing = required_cols - set(accuracy.columns)

    if missing:
        raise ValueError(
            f"Missing required columns in {accuracy_file}: {missing}\n"
            f"Columns found: {list(accuracy.columns)}"
        )

    accuracy = accuracy.copy()

    accuracy["stage_label"] = accuracy["Stage"].apply(normalize_stage)

    print(config["title"], "stage labels found:")
    print(
        accuracy[["Stage", "stage_label"]]
        .drop_duplicates()
        .sort_values(["stage_label", "Stage"])
        .to_string(index=False)
    )

    accuracy = accuracy[accuracy["stage_label"].isin(stage_order)].copy()

    if accuracy.empty:
        raise ValueError(
            f"No Stage 1 / Stage 2 / Overall rows found in {accuracy_file}"
        )

    agg_dict = {
        "mean_accuracy": ("accuracy", "mean"),
        "sd_accuracy": ("accuracy", "std"),
        "n_runs": ("accuracy", "size"),
    }

    # Only include extra aggregations if the column exists.
    for out_col, spec in config["extra_aggs"].items():
        in_col = spec[0]
        func = spec[1]

        if in_col in accuracy.columns:
            agg_dict[out_col] = (in_col, func)

    df = (
        accuracy
        .groupby([config["x_col"], "stage_label"], as_index=False)
        .agg(**agg_dict)
        .sort_values([config["x_col"], "stage_label"])
    )

    df["x_plot"] = config["x_transform"](df[config["x_col"]])

    df["sd_accuracy"] = df["sd_accuracy"].fillna(0)

    # Convert to percentage if accuracy is stored as 0-1.
    if df["mean_accuracy"].max() <= 1.5:
        df["acc_y"] = df["mean_accuracy"] * 100
        df["acc_sd"] = df["sd_accuracy"] * 100
    else:
        df["acc_y"] = df["mean_accuracy"]
        df["acc_sd"] = df["sd_accuracy"]

    return df


def get_secondary_labels(tick_df, config):
    key = config["key"]

    if key == "snps" and "mean_n_snps" in tick_df.columns:
        labels = [f"{int(round(v)):,}" for v in tick_df["mean_n_snps"]]
        label = "Mean SNPs retained"

    elif key == "escape" and "mean_n_escape_genes" in tick_df.columns:
        labels = [f"{int(round(v)):,}" for v in tick_df["mean_n_escape_genes"]]
        label = "Mean escape genes"

    elif key == "cells" and "mean_n_cells_after_cell_removal" in tick_df.columns:
        labels = [
            f"{int(round(v)):,}"
            for v in tick_df["mean_n_cells_after_cell_removal"]
        ]
        label = "Mean cells retained"

    elif key == "depth" and "mean_allele_reads_after" in tick_df.columns:
        labels = [format_reads(v) for v in tick_df["mean_allele_reads_after"]]
        label = "Allele reads retained"

    else:
        labels = [str(v) for v in tick_df["x_plot"]]
        label = ""

    return labels, label


def get_overall_rows_for_axis(df):
    """
    Use Overall rows for secondary-axis values.
    The extra quantities should be identical across stages, but using Overall
    avoids duplicate x positions.
    """
    overall = df[df["stage_label"] == "Overall"].copy()

    if overall.empty:
        overall = (
            df
            .drop_duplicates("x_plot")
            .sort_values("x_plot")
            .copy()
        )

    return overall


def add_secondary_axis_main(ax, df, config):
    if config["key"] == "skew":
        return None

    axis_df = get_overall_rows_for_axis(df)

    secax = ax.secondary_xaxis("bottom")

    ticks = config["x_ticks"]

    tick_df = (
        axis_df.iloc[[np.argmin(np.abs(axis_df["x_plot"] - t)) for t in ticks]]
        .drop_duplicates("x_plot")
        .sort_values("x_plot")
    )

    secax.set_xticks(tick_df["x_plot"])

    labels, label = get_secondary_labels(tick_df, config)

    secax.set_xticklabels(labels)
    secax.set_xlabel(label, fontsize=5.0, labelpad=1.7)
    secax.spines["bottom"].set_position(("outward", 19))
    secax.tick_params(
        axis="x",
        labelsize=4.5,
        pad=1.0,
        length=1.5,
        width=0.45,
    )

    return secax


def add_secondary_axis_zoom(ax, zoom_df, config):
    if config["key"] == "skew":
        return None

    axis_df = get_overall_rows_for_axis(zoom_df)

    if axis_df.empty:
        return None

    secax = ax.secondary_xaxis("bottom")

    ticks = config["zoom_ticks"]

    tick_df = (
        axis_df.iloc[[np.argmin(np.abs(axis_df["x_plot"] - t)) for t in ticks]]
        .drop_duplicates("x_plot")
        .sort_values("x_plot")
    )

    secax.set_xticks(tick_df["x_plot"])

    labels, label = get_secondary_labels(tick_df, config)

    secax.set_xticklabels(labels)
    secax.set_xlabel(label, fontsize=4.6, labelpad=1.2)
    secax.spines["bottom"].set_position(("outward", 13))
    secax.tick_params(
        axis="x",
        labelsize=4.1,
        pad=0.8,
        length=1.3,
        width=0.4,
    )

    return secax


# ============================================================
# Plotting
# ============================================================

def plot_accuracy_per_stage_across_sweeps():
    loaded = [(config, load_stage_accuracy_summary(config)) for config in simulations]

    fig = plt.figure(figsize=FIGSIZE)

    # One row per simulation, with small spacer rows between.
    height_ratios = []
    for i in range(len(simulations)):
        height_ratios.append(1.0)
        if i < len(simulations) - 1:
            if simulations[i]["key"] == "skew":
                height_ratios.append(0.36)
            else:
                height_ratios.append(0.55)

    gs = GridSpec(
        nrows=len(height_ratios),
        ncols=2,
        height_ratios=height_ratios,
        width_ratios=[2.65, 1.0],
        hspace=0.20,
        wspace=0.62,
    )

    for i, (config, df) in enumerate(loaded):
        row = i * 2

        ax_main = fig.add_subplot(gs[row, 0])
        ax_zoom = fig.add_subplot(gs[row, 1])

        # ----------------------------------------------------
        # Main panel
        # ----------------------------------------------------
        for stage in stage_order:
            sub = (
                df[df["stage_label"] == stage]
                .sort_values("x_plot")
                .copy()
            )

            if sub.empty:
                continue

            x = sub["x_plot"]
            y = sub["acc_y"]
            sd = sub["acc_sd"]

            col = stage_colours[stage]

            ax_main.plot(
                x,
                y,
                marker="o",
                linestyle=stage_linestyles[stage],
                color=col,
                label=stage,
            )

            ax_main.fill_between(
                x,
                y - sd,
                y + sd,
                color=col,
                alpha=0.13,
                linewidth=0,
            )

        ax_main.set_xlim(*config["xlim"])
        ax_main.set_ylim(0, 100)

        ax_main.set_xticks(config["x_ticks"])
        ax_main.set_xlabel(config["x_label"], labelpad=2.0, fontsize=5.7)

        ax_main.set_title(
            config["title"],
            loc="left",
            fontweight="bold",
            pad=1.6,
            fontsize=6.4,
        )

        add_secondary_axis_main(ax_main, df, config)

        # ----------------------------------------------------
        # Zoom panel: x between 99 and 100
        # ----------------------------------------------------
        zoom_df = df[df["x_plot"].between(99, 100)].copy()

        if zoom_df.empty:
            ax_zoom.set_visible(False)

        else:
            zoom_y = zoom_df["acc_y"]
            zoom_sd = zoom_df["acc_sd"]

            zoom_ymin, zoom_ymax = nice_zoom_limits(
                zoom_y,
                zoom_sd,
                lower_bound=0,
                upper_bound=100,
                step=1,
            )

            # Highlight zoom region on main panel.
            ax_main.add_patch(
                Rectangle(
                    (99, zoom_ymin),
                    1,
                    zoom_ymax - zoom_ymin,
                    facecolor="0.5",
                    edgecolor="0.35",
                    alpha=0.10,
                    linewidth=0.5,
                    zorder=0.5,
                )
            )

            for stage in stage_order:
                sub = (
                    zoom_df[zoom_df["stage_label"] == stage]
                    .sort_values("x_plot")
                    .copy()
                )

                if sub.empty:
                    continue

                xz = sub["x_plot"]
                yz = sub["acc_y"]
                sdz = sub["acc_sd"]

                col = stage_colours[stage]

                ax_zoom.plot(
                    xz,
                    yz,
                    marker="o",
                    linestyle=stage_linestyles[stage],
                    color=col,
                )

                ax_zoom.fill_between(
                    xz,
                    yz - sdz,
                    yz + sdz,
                    color=col,
                    alpha=0.13,
                    linewidth=0,
                )

            ax_zoom.set_xlim(98.95, 100.02)
            ax_zoom.set_ylim(zoom_ymin, zoom_ymax)
            ax_zoom.set_xticks(config["zoom_ticks"])

            ax_zoom.set_xlabel(
                config["x_label"],
                fontsize=5.0,
                labelpad=1.1,
            )

            ax_zoom.set_ylabel(
                "Accuracy (%)",
                fontsize=5.2,
                labelpad=1.0,
            )

            ax_zoom.tick_params(
                axis="x",
                labelsize=4.5,
                pad=0.8,
                width=0.5,
                length=1.7,
            )

            ax_zoom.tick_params(
                axis="y",
                labelsize=4.6,
                pad=0.8,
                width=0.5,
                length=1.7,
            )

            add_secondary_axis_zoom(ax_zoom, zoom_df, config)

            # Connector lines
            for y_main, y_zoom in [
                (zoom_ymin, ax_zoom.get_ylim()[0]),
                (zoom_ymax, ax_zoom.get_ylim()[1]),
            ]:
                con = ConnectionPatch(
                    xyA=(100, y_main),
                    coordsA=ax_main.transData,
                    xyB=(98.95, y_zoom),
                    coordsB=ax_zoom.transData,
                    color="0.35",
                    alpha=0.30,
                    linewidth=0.5,
                    zorder=0,
                    clip_on=False,
                )

                fig.add_artist(con)

        # ----------------------------------------------------
        # Styling
        # ----------------------------------------------------
        for ax in [ax_main, ax_zoom]:
            ax.spines["top"].set_visible(False)
            ax.grid(axis="y", alpha=0.25, linewidth=0.5)
            ax.set_axisbelow(True)

        ax_main.tick_params(
            axis="y",
            labelsize=5.4,
            pad=1.2,
            width=0.55,
            length=2.0,
        )

        ax_main.tick_params(
            axis="x",
            labelsize=5.4,
            pad=1.2,
            width=0.55,
            length=2.0,
        )

        if i == 2:
            ax_main.set_ylabel(
                "Accuracy (%)",
                fontsize=6.4,
                fontweight="bold",
                labelpad=5.0,
            )
        else:
            ax_main.set_ylabel("")

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------
    legend_handles = [
        Line2D(
            [0],
            [0],
            color=stage_colours[stage],
            marker="o",
            linestyle=stage_linestyles[stage],
            linewidth=1.0,
            markersize=2.8,
            label=stage,
        )
        for stage in stage_order
    ]

    fig.legend(
        legend_handles,
        stage_order,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.994),
        ncol=3,
        frameon=False,
        handlelength=1.2,
        columnspacing=0.9,
        labelspacing=0.25,
        fontsize=5.0,
    )

    fig.subplots_adjust(
        left=0.12,
        right=0.97,
        top=0.955,
        bottom=0.04,
    )

    save_all(
        fig,
        "combined_simulations_accuracy_per_stage_and_overall_80x180mm_sd",
    )


# ============================================================
# Make figure
# ============================================================

plot_accuracy_per_stage_across_sweeps()

print(f"Wrote stage accuracy sweep plot to: {plot_dir}")

Reading: /home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/skew/skew_accuracy_summary_by_run_stage.tsv
Skew stage labels found:
Stage stage_label
  all     Overall
  1.0     Stage 1
  2.0     Stage 2
Reading: /home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/snp_removal2/snp_removal_accuracy_summary_by_run_stage.tsv
SNP removal stage labels found:
Stage stage_label
  all     Overall
  1.0     Stage 1
  2.0     Stage 2
Reading: /home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/escape/escape_accuracy_summary_by_run_stage.tsv
Escape stage labels found:
Stage stage_label
  all     Overall
  1.0     Stage 1
  2.0     Stage 2
Reading: /home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/cell_removal/cell_removal_accuracy_summary_by_run_stage.tsv
Cell removal stage labels found:
Stage stage_label
  all     Overall
  1.0     Stage 1
  2.0     Stage 2
Reading: /home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/read_removal/read_removal_accuracy

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote stage accuracy sweep plot to: /home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/simulation_combined/plots


In [7]:
# ============================================================
# Horizontal combined faceted simulation plot:
# accuracy per stage + number of SNPs per stage across sweeps
#
# Layout:
#   columns = skew, SNP removal, escape, cells, depth
#   row 1   = accuracy per stage + overall
#   row 2   = mean number of SNPs per stage + overall
#
# Lines:
#   Stage 1
#   Stage 2
#   Overall
#
# Values are plotted as mean ± SD across runs.
# ============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter


# ============================================================
# Paths
# ============================================================

plot_dir = Path(
    "/home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/simulation_combined/plots"
)
plot_dir.mkdir(parents=True, exist_ok=True)

formats = ["pdf", "svg", "png", "eps"]


# ============================================================
# Figure sizing
# ============================================================

MM_PER_INCH = 25.4

# Horizontal version of the previous 80 x 180 mm figure.
FIG_WIDTH_MM = 180
FIG_HEIGHT_MM = 80

FIGSIZE = (FIG_WIDTH_MM / MM_PER_INCH, FIG_HEIGHT_MM / MM_PER_INCH)


# ============================================================
# Matplotlib style
# ============================================================

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 6.0,
    "axes.labelsize": 6.0,
    "axes.titlesize": 6.8,
    "xtick.labelsize": 5.0,
    "ytick.labelsize": 5.0,
    "legend.fontsize": 5.4,

    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
    "lines.linewidth": 1.0,
    "lines.markersize": 2.5,
})


# ============================================================
# Colours
# ============================================================

stage_order = ["Stage 1", "Stage 2", "Overall"]

stage_colours = {
    "Stage 1": "#4C78A8",
    "Stage 2": "#F58518",
    "Overall": "#E76F51",
}

stage_linestyles = {
    "Stage 1": "-",
    "Stage 2": "-",
    "Overall": "-",
}


# ============================================================
# Helpers
# ============================================================

def save_all(fig, name):
    for fmt in formats:
        fig.savefig(
            plot_dir / f"{name}.{fmt}",
            dpi=300,
            bbox_inches="tight",
            transparent=True,
        )
    plt.close(fig)


def format_count_axis(x, pos=None):
    x = float(x)

    if abs(x) >= 1_000_000:
        return f"{x / 1_000_000:.1f}M"

    if abs(x) >= 10_000:
        return f"{x / 1_000:.0f}k"

    if abs(x) >= 1_000:
        return f"{x / 1_000:.1f}k"

    return f"{int(round(x))}"


def normalize_stage(stage):
    """
    Robustly convert stage labels to:
        Stage 1
        Stage 2
        Overall

    Handles examples like:
        1
        1.0
        Stage 1
        Stage 1.0
        stage_1
        all
        overall
    """

    s = str(stage).strip().lower()
    s = s.replace("_", " ").replace("-", " ")

    if s in {"all", "overall", "stage all"}:
        return "Overall"

    if s in {"1", "1.0"}:
        return "Stage 1"

    if s in {"2", "2.0"}:
        return "Stage 2"

    nums = re.findall(r"\d+(?:\.\d+)?", s)

    if "stage" in s and nums:
        stage_num = int(float(nums[0]))

        if stage_num == 1:
            return "Stage 1"

        if stage_num == 2:
            return "Stage 2"

    return str(stage)


# ============================================================
# Simulation definitions
# ============================================================

simulations = [
    {
        "key": "skew",
        "title": "Skew",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/skew"),
        "accuracy_file": "skew_accuracy_summary_by_run_stage.tsv",
        "x_col": "x1_fraction_setting",
        "x_label": "X1 fraction (%)",
        "x_transform": lambda s: s * 100,
        "x_ticks": [50, 75, 100],
        "xlim": (50, 100),
    },
    {
        "key": "snps",
        "title": "SNP removal",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/snp_removal2"),
        "accuracy_file": "snp_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "snp_removal_pct_setting",
        "x_label": "SNPs removed (%)",
        "x_transform": lambda s: s,
        "x_ticks": [0, 50, 100],
        "xlim": (0, 100),
    },
    {
        "key": "escape",
        "title": "Escape",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/escape"),
        "accuracy_file": "escape_accuracy_summary_by_run_stage.tsv",
        "x_col": "escape_pct_setting",
        "x_label": "Escape genes (%)",
        "x_transform": lambda s: s,
        "x_ticks": [0, 50, 100],
        "xlim": (0, 100),
    },
    {
        "key": "cells",
        "title": "Cell removal",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/cell_removal"),
        "accuracy_file": "cell_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "cell_removal_pct_setting",
        "x_label": "Cells removed (%)",
        "x_transform": lambda s: s,
        "x_ticks": [0, 50, 100],
        "xlim": (0, 100),
    },
    {
        "key": "depth",
        "title": "Depth",
        "base": Path("/home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/read_removal"),
        "accuracy_file": "read_removal_accuracy_summary_by_run_stage.tsv",
        "x_col": "read_removal_pct_setting",
        "x_label": "Reads removed (%)",
        "x_transform": lambda s: s,
        "x_ticks": [0, 50, 100],
        "xlim": (0, 100),
    },
]


# ============================================================
# Loading and aggregation
# ============================================================

def load_stage_accuracy_summary(config):
    accuracy_file = config["base"] / config["accuracy_file"]

    print(f"Reading: {accuracy_file}")

    accuracy = pd.read_csv(accuracy_file, sep="\t")

    required_cols = {config["x_col"], "Stage", "accuracy"}
    missing = required_cols - set(accuracy.columns)

    if missing:
        raise ValueError(
            f"Missing required columns in {accuracy_file}: {missing}\n"
            f"Columns found: {list(accuracy.columns)}"
        )

    accuracy = accuracy.copy()
    accuracy["stage_label"] = accuracy["Stage"].apply(normalize_stage)

    print(config["title"], "stage labels found:")
    print(
        accuracy[["Stage", "stage_label"]]
        .drop_duplicates()
        .sort_values(["stage_label", "Stage"])
        .to_string(index=False)
    )

    accuracy = accuracy[accuracy["stage_label"].isin(stage_order)].copy()

    if accuracy.empty:
        raise ValueError(
            f"No Stage 1 / Stage 2 / Overall rows found in {accuracy_file}"
        )

    agg_dict = {
        "mean_accuracy": ("accuracy", "mean"),
        "sd_accuracy": ("accuracy", "std"),
        "n_runs": ("accuracy", "size"),
    }

    # Key change: aggregate n_snps by x-value AND stage, so the bottom row
    # shows how many SNPs are present in Stage 1, Stage 2 and Overall.
    if "n_snps" in accuracy.columns:
        agg_dict["mean_n_snps"] = ("n_snps", "mean")
        agg_dict["sd_n_snps"] = ("n_snps", "std")
    else:
        print(
            f"Warning: no 'n_snps' column found in {accuracy_file}; "
            "the SNP-count panel for this sweep will be blank."
        )

    df = (
        accuracy
        .groupby([config["x_col"], "stage_label"], as_index=False)
        .agg(**agg_dict)
        .sort_values([config["x_col"], "stage_label"])
    )

    df["x_plot"] = config["x_transform"](df[config["x_col"]])

    df["sd_accuracy"] = df["sd_accuracy"].fillna(0)

    if "sd_n_snps" in df.columns:
        df["sd_n_snps"] = df["sd_n_snps"].fillna(0)

    # Convert to percentage if accuracy is stored as 0-1.
    if df["mean_accuracy"].max() <= 1.5:
        df["acc_y"] = df["mean_accuracy"] * 100
        df["acc_sd"] = df["sd_accuracy"] * 100
    else:
        df["acc_y"] = df["mean_accuracy"]
        df["acc_sd"] = df["sd_accuracy"]

    return df


# ============================================================
# Plotting helpers
# ============================================================

def plot_accuracy_panel(ax, df, config):
    for stage in stage_order:
        sub = (
            df[df["stage_label"] == stage]
            .sort_values("x_plot")
            .copy()
        )

        if sub.empty:
            continue

        x = sub["x_plot"].to_numpy(dtype=float)
        y = sub["acc_y"].to_numpy(dtype=float)
        sd = sub["acc_sd"].to_numpy(dtype=float)

        col = stage_colours[stage]

        ax.plot(
            x,
            y,
            marker="o",
            linestyle=stage_linestyles[stage],
            color=col,
            label=stage,
        )

        ax.fill_between(
            x,
            np.maximum(y - sd, 0),
            np.minimum(y + sd, 100),
            color=col,
            alpha=0.13,
            linewidth=0,
        )

    ax.set_xlim(*config["xlim"])
    ax.set_ylim(0, 100)
    ax.set_xticks(config["x_ticks"])

    # The x-axis label is shown on the SNP row below each panel.
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelbottom=False)

    ax.set_title(
        config["title"],
        loc="left",
        fontweight="bold",
        pad=1.8,
        fontsize=6.6,
    )


def plot_snp_count_panel(ax, df, config):
    if "mean_n_snps" not in df.columns:
        ax.text(
            0.5,
            0.5,
            "n_snps\nnot found",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=5.2,
        )
        ax.set_xlim(*config["xlim"])
        ax.set_xticks(config["x_ticks"])
        ax.set_xlabel(config["x_label"], labelpad=1.8, fontsize=5.4)
        return

    ymax = 0.0

    for stage in stage_order:
        sub = (
            df[df["stage_label"] == stage]
            .sort_values("x_plot")
            .copy()
        )

        if sub.empty:
            continue

        x = sub["x_plot"].to_numpy(dtype=float)
        y = sub["mean_n_snps"].to_numpy(dtype=float)
        sd = sub["sd_n_snps"].to_numpy(dtype=float)

        col = stage_colours[stage]

        ax.plot(
            x,
            y,
            marker="o",
            linestyle=stage_linestyles[stage],
            color=col,
            label=stage,
        )

        ax.fill_between(
            x,
            np.maximum(y - sd, 0),
            y + sd,
            color=col,
            alpha=0.13,
            linewidth=0,
        )

        ymax = max(ymax, float(np.nanmax(y + sd)))

    ax.set_xlim(*config["xlim"])
    ax.set_xticks(config["x_ticks"])
    ax.set_xlabel(config["x_label"], labelpad=1.8, fontsize=5.4)

    ax.set_ylim(0, ymax * 1.08 if ymax > 0 else 1)
    ax.yaxis.set_major_formatter(FuncFormatter(format_count_axis))


def style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.25, linewidth=0.5)
    ax.set_axisbelow(True)

    ax.tick_params(
        axis="y",
        labelsize=5.0,
        pad=1.0,
        width=0.55,
        length=2.0,
    )

    ax.tick_params(
        axis="x",
        labelsize=5.0,
        pad=1.0,
        width=0.55,
        length=2.0,
    )


# ============================================================
# Main plot
# ============================================================

def plot_accuracy_and_stage_snps_across_sweeps_horizontal():
    loaded = [(config, load_stage_accuracy_summary(config)) for config in simulations]

    fig = plt.figure(figsize=FIGSIZE)

    gs = GridSpec(
        nrows=2,
        ncols=len(simulations),
        height_ratios=[1.0, 1.0],
        width_ratios=[1.0] * len(simulations),
        hspace=0.34,
        wspace=0.42,
    )

    for i, (config, df) in enumerate(loaded):
        ax_acc = fig.add_subplot(gs[0, i])
        ax_snps = fig.add_subplot(gs[1, i])

        plot_accuracy_panel(ax_acc, df, config)
        plot_snp_count_panel(ax_snps, df, config)

        style_axis(ax_acc)
        style_axis(ax_snps)

        # Accuracy is on the same scale for all columns, so only label the first.
        if i == 0:
            ax_acc.set_ylabel(
                "Accuracy (%)",
                fontsize=6.2,
                fontweight="bold",
                labelpad=3.5,
            )
            ax_snps.set_ylabel(
                "Mean SNPs",
                fontsize=6.2,
                fontweight="bold",
                labelpad=3.5,
            )
        else:
            ax_acc.set_ylabel("")
            ax_acc.tick_params(axis="y", labelleft=False)
            ax_snps.set_ylabel("")
            # Keep SNP-count y tick labels because each sweep may have its own scale.

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------
    legend_handles = [
        Line2D(
            [0],
            [0],
            color=stage_colours[stage],
            marker="o",
            linestyle=stage_linestyles[stage],
            linewidth=1.0,
            markersize=2.8,
            label=stage,
        )
        for stage in stage_order
    ]

    fig.legend(
        legend_handles,
        stage_order,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.985),
        ncol=3,
        frameon=False,
        handlelength=1.2,
        columnspacing=1.0,
        labelspacing=0.25,
        fontsize=5.4,
    )

    fig.subplots_adjust(
        left=0.055,
        right=0.995,
        top=0.860,
        bottom=0.155,
    )

    save_all(
        fig,
        "combined_simulations_accuracy_per_stage_and_stage_snps_horizontal_180x80mm_sd",
    )


# ============================================================
# Make figure
# ============================================================

plot_accuracy_and_stage_snps_across_sweeps_horizontal()

print(f"Wrote horizontal stage accuracy + SNP-count sweep plot to: {plot_dir}")


Reading: /home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/skew/skew_accuracy_summary_by_run_stage.tsv
Skew stage labels found:
Stage stage_label
  all     Overall
  1.0     Stage 1
  2.0     Stage 2
Reading: /home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/snp_removal2/snp_removal_accuracy_summary_by_run_stage.tsv
SNP removal stage labels found:
Stage stage_label
  all     Overall
  1.0     Stage 1
  2.0     Stage 2
Reading: /home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/escape/escape_accuracy_summary_by_run_stage.tsv
Escape stage labels found:
Stage stage_label
  all     Overall
  1.0     Stage 1
  2.0     Stage 2
Reading: /home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/cell_removal/cell_removal_accuracy_summary_by_run_stage.tsv
Cell removal stage labels found:
Stage stage_label
  all     Overall
  1.0     Stage 1
  2.0     Stage 2
Reading: /home/913/dk4874/scratch/vn68_gdata/scDaisychain_simulations/read_removal/read_removal_accuracy

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote horizontal stage accuracy + SNP-count sweep plot to: /home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/simulation_combined/plots
